# 01 · Data Loading

Builds `combined_df` from raw sales CSVs and auxiliary enrichment files,  
then persists the result to `outputs/combined_df.parquet`.

All downstream notebooks load directly from the Parquet file — this notebook runs once.

```
inventory-analysis/
├── notebooks/  01_data_loading.ipynb
├── src/        data_loader.py
└── outputs/    combined_df.parquet
```

## 0 · Dependencies

In [2]:
!pip install pandas numpy pyarrow openpyxl -q

## 1 · Imports

In [13]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import data_loader as dl

## 2 · Configuration

Override `BASE_DIR` here if needed. All sub-paths are derived from it.

In [ ]:
# dl.BASE_DIR    = Path(r"D:\your\project\path")
# dl.SALES_DIR   = dl.BASE_DIR / "Sales_data"
# dl.HELPERS_DIR = dl.BASE_DIR / "Helping_Files"
# dl.OUTPUT_DIR  = PROJECT_ROOT / "outputs"

print("sales dir  :", dl.SALES_DIR)
print("helpers dir:", dl.HELPERS_DIR)
print("output dir :", dl.OUTPUT_DIR)

## 3 · Run pipeline

In [15]:
combined_df = dl.run()


── Stage 1 · Sales Ingestion ─────────────────────────────────
  records : 4,151,887  (excluded 456,012)
  range   : 2019-01-01 → 2025-12-20
  skus    : 62,713

── Stage 2 · Cost Assignment & Profit ────────────────────────
  profit rows : 4,151,661
  total profit: 769,193,376 SAR
  avg margin  : 50.7%

── Stage 3 · Enrichment Loading ──────────────────────────────

── Stage 4 · Merge ───────────────────────────────────────────
  [historical_qty] outer join → 4,222,805 rows (+70,918)

── Stage 5 · Cleaning ────────────────────────────────────────
  shape  : (4222805, 40)
  memory : 2475.9 MB

── Stage 6 · Save ────────────────────────────────────────────
  D:\Projects\InventoryDeepDive\Outputs\combined_df.parquet  (104.5 MB)


## 4 · Verification

### Shape & schema

In [14]:
print(f"shape      : {combined_df.shape}")
print(f"date range : {combined_df['Date'].min().date()} → {combined_df['Date'].max().date()}")
print(f"unique skus: {combined_df['SKU'].nunique():,}")
print(f"memory     : {combined_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
combined_df.dtypes

shape      : (4222805, 40)
date range : 2019-01-01 → 2025-12-20
unique skus: 133,631
memory     : 2475.9 MB


Section                              string
Section Name                         string
bal Value                           float32
bal Qty                             float32
U Price                             float32
Client                               string
Outlet                              float32
Outlet Name                          string
Date                         datetime64[us]
SKU                                  string
Catalog No.                          string
Year                                  Int64
Total_Cost                          float64
Total_Profit                        float64
Profit_Margin_%                     float64
CURRENT_STOCK                       float64
OUTSTANDING                         float64
PR                                  float64
EFFECTIVE_STOCK                     float64
CATEGORY                             string
Supplier                             string
LAST_ENTRY_DATE              datetime64[us]
Nb. Days (Avail. Balance)       

### Null coverage

In [6]:
(
    combined_df.isna()
    .mean()
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
    .to_frame("null_%")
    .query("`null_%` > 0")
)

,null_%
SERIAL,36.4
Catalog No.,4.9
COLLECTION_NAME,3.1
CATALOG_NO,2.8
TEXTURE,2.1
COLOR_NAME,2.1
SUB_CATEGORY,1.9
ARTPATERN,1.9
IS_PLAIN_SECTION,1.9
Outlet,1.7


### Financial summary by year

In [7]:
(
    combined_df
    .groupby("Year", observed=True)
    .agg(
        Revenue      = ("bal Value",       "sum"),
        Total_Profit = ("Total_Profit",    "sum"),
        Avg_Margin   = ("Profit_Margin_%", "mean"),
        Transactions = ("SKU",             "count"),
        Unique_SKUs  = ("SKU",             "nunique"),
    )
    .round({"Avg_Margin": 1})
)

,Revenue,Total_Profit,Avg_Margin,Transactions,Unique_SKUs
Year,,,,,
2019,332473184.0,1.921094e+08,NaN,1087892,43870
2020,275161696.0,1.623715e+08,NaN,915049,42848
2021,231364352.0,1.423747e+08,NaN,762663,42203
2022,148481440.0,9.004910e+07,NaN,477259,36164
2023,126325704.0,5.557349e+07,NaN,399148,33980
2024,105613728.0,5.957203e+07,NaN,257225,32900
2025,113589432.0,6.714313e+07,NaN,252602,32313


### Enrichment coverage

In [8]:
enrichment_cols = [
    "CATEGORY", "Supplier", "CURRENT_STOCK", "LAST_ENTRY_DATE",
    "Nb. Days (Avail. Balance)", "COST_PER_DOLLAR", "COLOR_NAME",
    "ARTPATERN", "IS_PLAIN_SECTION", "CATALOG_NO", "COLLECTION_NAME",
    "Outlet_Type", "Area_Master_Category", "Country",
    "First_Inv_Date", "OLD_QTY",
]

present = [c for c in enrichment_cols if c in combined_df.columns]
(
    combined_df[present]
    .notna()
    .mean()
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
    .to_frame("coverage_%")
)

,coverage_%
First_Inv_Date,100.0
OLD_QTY,99.2
CATEGORY,98.3
Supplier,98.3
Nb. Days (Avail. Balance),98.3
COST_PER_DOLLAR,98.3
CURRENT_STOCK,98.3
LAST_ENTRY_DATE,98.3
Area_Master_Category,98.3
Country,98.3


### Sales type split

In [9]:
combined_df["Sales_Type"].value_counts().to_frame("count")

,count
Sales_Type,
FAMILY,4193305
CONTRACT,29500


## 5 · Load in downstream notebooks

```python
import pandas as pd
combined_df = pd.read_parquet("../outputs/combined_df.parquet")
```

In [10]:
parquet_path = PROJECT_ROOT / "outputs" / "combined_df.parquet"
test = pd.read_parquet(parquet_path)
assert test.shape == combined_df.shape
print(f"parquet OK · {test.shape} · {parquet_path.stat().st_size / 1024**2:.1f} MB")
del test

parquet OK · (4222805, 40) · 104.5 MB
